# 10 — LLMs + Pydantic AI: Structured Outputs from Language Models

## What This Notebook Covers

Large Language Models (LLMs) like Llama 3 can answer almost any question —  
but by default they return *free text*.  For real applications we often want  
**structured, typed data** that our code can work with reliably.

**Pydantic AI** is a Python library that lets you:
- Talk to an LLM with a simple Agent API
- Declare a Pydantic model as the *expected output type*
- Receive a validated Python object (not a raw string)

We use **Ollama** to run Llama 3.2 locally (no API key required).

> **Prerequisite:** Ollama must be running locally.  
> Start it with: `ollama serve` and ensure `llama3.2` is pulled.

## Setup

In [ ]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider

MODEL_NAME = 'llama3.2'

# Connect to the locally-running Ollama server
ollama_model = OpenAIChatModel(
    model_name=MODEL_NAME,
    provider=OllamaProvider(base_url='http://localhost:11434/v1'),
)

print(f'Using model: {MODEL_NAME} via Ollama')

## Demo 1 — Plain Text Response

The simplest usage: send a question, get a text answer.

In [ ]:
plain_agent = Agent(ollama_model)

result = plain_agent.run_sync(
    'Explain what a neural network is in 2 sentences, as if talking to a 10-year-old.'
)

print('Answer:')
print(result.output)

## Demo 2 — Structured Output with Pydantic

Instead of free text, we define a Pydantic model that describes exactly  
what fields we want — and Pydantic AI ensures the LLM fills them all in.

This is powerful for building apps: you can pass `explanation.difficulty`  
to filter content, or display `explanation.real_world_analogy` in a UI.

In [ ]:
class ConceptExplanation(BaseModel):
    concept:           str = Field(description="The AI concept being explained")
    one_line_summary:  str = Field(description="What it is in one sentence")
    real_world_analogy: str = Field(description="An analogy a non-technical person would understand")
    example:           str = Field(description="A concrete example (2-3 sentences)")
    difficulty:        str = Field(description="Beginner / Intermediate / Advanced")


structured_agent = Agent(ollama_model, output_type=ConceptExplanation)

concepts = ['gradient descent', 'overfitting', 'neural network']

for concept in concepts:
    result = structured_agent.run_sync(
        f'Explain the following AI concept for a beginner: {concept}'
    )
    e = result.output
    print(f'--- {e.concept} ({e.difficulty}) ---')
    print(f'Summary : {e.one_line_summary}')
    print(f'Analogy : {e.real_world_analogy}')
    print(f'Example : {e.example}')
    print()

## Demo 3 — Quiz Question Generator

We define a `QuizQuestion` model and give the agent a system prompt  
that shapes its personality.  Each call returns a ready-to-use quiz question  
with options, the correct answer index, and an explanation.

In [ ]:
class QuizQuestion(BaseModel):
    question:             str
    options:              list[str] = Field(description='Exactly 4 multiple choice options')
    correct_option_index: int       = Field(description='0-based index of the correct option')
    explanation:          str


quiz_agent = Agent(
    ollama_model,
    output_type=QuizQuestion,
    system_prompt=(
        'You are a friendly AI teacher. '
        'Generate short, clear multiple-choice quiz questions '
        'about basic AI and machine learning concepts.'
    ),
)

topics = ['What gradient descent does', 'What overfitting means']

for topic in topics:
    result = quiz_agent.run_sync(f'Create a quiz question about: {topic}')
    q = result.output
    print(f'Q: {q.question}')
    for i, opt in enumerate(q.options):
        marker = '✓' if i == q.correct_option_index else ' '
        print(f'  [{marker}] {i + 1}. {opt}')
    print(f'  → {q.explanation}')
    print()

## Key Lessons

1. **LLMs produce text** — but with Pydantic AI you can get **typed Python objects**.
2. **`output_type=MyModel`** tells the agent what schema to fill in.  
   Under the hood it uses structured outputs / function calling.
3. **System prompts** shape the model's personality and focus.
4. **Ollama** lets you run powerful open models locally — no API key, no cost.
5. This pattern (define schema → call LLM → use structured data) is increasingly  
   common in production AI applications.